# 05 — Final Model Evaluation
## AT&T Spam Detector

### Objective

This notebook performs the final evaluation of the frozen spam-classification candidates on the held-out test set.

All architecture choices, preprocessing decisions, and operating thresholds were selected using training and validation data only.

The test set is used here for the first time to estimate final generalization performance.

The evaluated models are:

1. embedding + global-average-pooling baseline;
2. embedding + GRU sequence model;
3. Universal Sentence Encoder transfer-learning classifier.

No model selection, threshold tuning, or retraining is performed using test results.

In [ ]:
# ---------------------------------------------------------------------------
# Imports and reproducibility
# ---------------------------------------------------------------------------

from pathlib import Path

import numpy as np
import pandas as pd
import tensorflow as tf
import tensorflow_hub as hub

from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
    average_precision_score,
)

RANDOM_STATE = 42

np.random.seed(RANDOM_STATE)
tf.random.set_seed(RANDOM_STATE)

print(f"TensorFlow version:     {tf.__version__}")
print(f"TensorFlow Hub version: {hub.__version__}")
print(f"NumPy version:          {np.__version__}")
print(f"Pandas version:         {pd.__version__}")

### 5.1 Load the held-out test set

The test split was created before model development and has remained excluded from architecture selection, training decisions, and threshold tuning.

It is now used once for final generalization evaluation.

In [ ]:
# ---------------------------------------------------------------------------
# Load the held-out test split
# ---------------------------------------------------------------------------

PROJECT_ROOT = Path.cwd().parent
TEST_PATH = PROJECT_ROOT / "data" / "processed" / "test.csv"

test_df = pd.read_csv(TEST_PATH)

X_test = test_df["text"].astype(str)

label_mapping = {
    "ham": 0,
    "spam": 1,
}

y_test = test_df["label"].map(label_mapping)

print(f"Test shape: {test_df.shape}")

print("\nTest class counts:")
print(y_test.value_counts().sort_index())

print(
    f"\nMissing labels: {y_test.isna().sum()}"
)

### 5.2 Reconstruct the frozen word-level preprocessing

The baseline and GRU models were trained with the same `TextVectorization` configuration.

The vectorizer is reconstructed using the training split only, preserving the original leakage-free workflow. The test set is transformed only after the vocabulary has been adapted on training data.

A custom standardization function ensures that punctuation-only SMS messages are mapped to a dedicated `[EMPTY]` token rather than becoming fully padded sequences.

In [ ]:
# ---------------------------------------------------------------------------
# Reconstruct the training-only TextVectorization pipeline
# ---------------------------------------------------------------------------

TRAIN_PATH = PROJECT_ROOT / "data" / "processed" / "train.csv"

train_df = pd.read_csv(TRAIN_PATH)

X_train = train_df["text"].astype(str)

SEQUENCE_LENGTH = 50


def safe_standardize(text):
    """
    Lowercase text, strip punctuation, and replace empty results
    with a dedicated token so that no SMS becomes fully padded.
    """
    text = tf.strings.lower(text)

    text = tf.strings.regex_replace(
        text,
        r"[!\"#$%&'()*+,-./:;<=>?@\[\\\]^_`{|}~]",
        "",
    )

    text = tf.strings.strip(text)

    text = tf.where(
        tf.strings.length(text) > 0,
        text,
        "[EMPTY]",
    )

    return text


text_vectorizer = tf.keras.layers.TextVectorization(
    standardize=safe_standardize,
    split="whitespace",
    output_mode="int",
    output_sequence_length=SEQUENCE_LENGTH,
)

# Adapt only on the training split to prevent information leakage.
text_vectorizer.adapt(X_train)

VOCAB_SIZE = len(text_vectorizer.get_vocabulary())

print(f"Training messages: {len(X_train)}")
print(f"Vocabulary size:   {VOCAB_SIZE}")
print(f"Sequence length:   {SEQUENCE_LENGTH}")

In [ ]:
# ---------------------------------------------------------------------------
# Transform the held-out test messages
# ---------------------------------------------------------------------------

X_test_vectorized = text_vectorizer(
    tf.constant(X_test.to_numpy())
).numpy()

print(f"Vectorized test shape: {X_test_vectorized.shape}")

fully_padded_test = np.sum(
    np.all(X_test_vectorized == 0, axis=1)
)

print(f"Fully padded test messages: {fully_padded_test}")

### 5.3 Load the frozen baseline and GRU models

The baseline and GRU models were trained and selected using training and validation data only.

Their operating thresholds were frozen during validation:

- baseline threshold: `0.40`;
- GRU threshold: `0.89`.

The test set is used only to measure final generalization performance.

In [ ]:
# ---------------------------------------------------------------------------
# Load the frozen word-level models
# ---------------------------------------------------------------------------

MODELS_DIR = PROJECT_ROOT / "models"

BASELINE_MODEL_PATH = MODELS_DIR / "baseline_embedding_model.keras"
GRU_MODEL_PATH = MODELS_DIR / "gru_sequence_model.keras"

baseline_model = tf.keras.models.load_model(BASELINE_MODEL_PATH)
gru_model = tf.keras.models.load_model(GRU_MODEL_PATH)

BASELINE_THRESHOLD = 0.40
GRU_THRESHOLD = 0.89

print(f"Baseline model exists: {BASELINE_MODEL_PATH.exists()}")
print(f"GRU model exists:      {GRU_MODEL_PATH.exists()}")

print(f"\nBaseline threshold: {BASELINE_THRESHOLD:.2f}")
print(f"GRU threshold:      {GRU_THRESHOLD:.2f}")

In [ ]:
# ---------------------------------------------------------------------------
# Generate held-out test probabilities
# ---------------------------------------------------------------------------

baseline_test_probabilities = baseline_model.predict(
    X_test_vectorized,
    verbose=0,
).ravel()

gru_test_probabilities = gru_model.predict(
    X_test_vectorized,
    verbose=0,
).ravel()

print(
    "Baseline probability range: "
    f"{baseline_test_probabilities.min():.6f} "
    f"to {baseline_test_probabilities.max():.6f}"
)

print(
    "GRU probability range: "
    f"{gru_test_probabilities.min():.6f} "
    f"to {gru_test_probabilities.max():.6f}"
)

print(f"\nBaseline predictions: {len(baseline_test_probabilities)}")
print(f"GRU predictions:      {len(gru_test_probabilities)}")

### 5.4 Final test evaluation — baseline and GRU

The following metrics are computed using the operating thresholds selected previously on the validation set.

No threshold optimization is performed on the test set.

Because spam is the positive class and the dataset is imbalanced, precision, recall, F1-score, ROC-AUC, PR-AUC, false positives, and false negatives are reported alongside accuracy.

In [ ]:
# ---------------------------------------------------------------------------
# Helper function for final binary-classification evaluation
# ---------------------------------------------------------------------------

def evaluate_test_predictions(
    model_name,
    y_true,
    probabilities,
    threshold,
):
    """
    Evaluate a frozen binary classifier at its pre-selected threshold.

    The threshold must have been chosen before test evaluation.
    """
    predictions = (probabilities >= threshold).astype(int)

    cm = confusion_matrix(y_true, predictions)

    tn, fp, fn, tp = cm.ravel()

    metrics = {
        "model": model_name,
        "threshold": threshold,
        "accuracy": accuracy_score(y_true, predictions),
        "precision": precision_score(y_true, predictions),
        "recall": recall_score(y_true, predictions),
        "f1": f1_score(y_true, predictions),
        "roc_auc": roc_auc_score(y_true, probabilities),
        "pr_auc": average_precision_score(y_true, probabilities),
        "false_positives": int(fp),
        "false_negatives": int(fn),
    }

    return metrics, predictions, cm

In [ ]:
# ---------------------------------------------------------------------------
# Evaluate baseline and GRU at their frozen validation thresholds
# ---------------------------------------------------------------------------

baseline_test_metrics, baseline_test_predictions, baseline_test_cm = (
    evaluate_test_predictions(
        model_name="Baseline",
        y_true=y_test,
        probabilities=baseline_test_probabilities,
        threshold=BASELINE_THRESHOLD,
    )
)

gru_test_metrics, gru_test_predictions, gru_test_cm = (
    evaluate_test_predictions(
        model_name="GRU",
        y_true=y_test,
        probabilities=gru_test_probabilities,
        threshold=GRU_THRESHOLD,
    )
)

word_model_test_results = pd.DataFrame(
    [
        baseline_test_metrics,
        gru_test_metrics,
    ]
)

word_model_test_results.round(4)

In [ ]:
# ---------------------------------------------------------------------------
# Inspect final confusion matrices
# ---------------------------------------------------------------------------

print("Baseline confusion matrix:")
print(baseline_test_cm)

print("\nGRU confusion matrix:")
print(gru_test_cm)

### 5.5 Final test evaluation — Universal Sentence Encoder

The Universal Sentence Encoder (USE) model was selected as the leading candidate based exclusively on validation performance.

Unlike the word-level baseline and GRU models, USE operates directly on raw SMS text using a pretrained 512-dimensional semantic representation.

The pretrained encoder remains frozen. The saved dense classification head is loaded and evaluated at its validation-selected threshold of `0.50`.

No adaptation, retraining, or threshold optimization is performed on the test set.

In [ ]:
# ---------------------------------------------------------------------------
# Load the frozen USE encoder and trained classification head
# ---------------------------------------------------------------------------

USE_URL = "https://tfhub.dev/google/universal-sentence-encoder/4"

USE_CLASSIFIER_PATH = (
    MODELS_DIR / "use_transfer_classifier.keras"
)

USE_THRESHOLD = 0.50

# Load the same pretrained sentence encoder used during development.
use_encoder = hub.load(USE_URL)

# Load the classification head trained on frozen USE representations.
use_classifier = tf.keras.models.load_model(
    USE_CLASSIFIER_PATH
)

print(f"USE classifier exists: {USE_CLASSIFIER_PATH.exists()}")
print(f"USE threshold:         {USE_THRESHOLD:.2f}")

In [ ]:
# ---------------------------------------------------------------------------
# Encode held-out SMS messages with the frozen pretrained USE encoder
# ---------------------------------------------------------------------------

def encode_messages(messages, batch_size=64):
    """
    Encode SMS messages into frozen 512-dimensional Universal Sentence
    Encoder representations without fitting on the evaluation data.
    """
    dataset = tf.data.Dataset.from_tensor_slices(
        messages.to_numpy()
    ).batch(batch_size)

    embedding_batches = []

    for batch in dataset:
        batch_embeddings = use_encoder(batch)
        embedding_batches.append(
            batch_embeddings.numpy()
        )

    return np.concatenate(
        embedding_batches,
        axis=0,
    )


X_test_use = encode_messages(X_test)

print(f"USE test embedding shape: {X_test_use.shape}")
print(f"Embedding dtype:          {X_test_use.dtype}")
print(f"Contains NaN:             {np.isnan(X_test_use).any()}")
print(f"Contains infinity:        {np.isinf(X_test_use).any()}")

In [ ]:
# ---------------------------------------------------------------------------
# Generate USE probabilities and perform final held-out evaluation
# ---------------------------------------------------------------------------

use_test_probabilities = use_classifier.predict(
    X_test_use,
    verbose=0,
).ravel()

use_test_metrics, use_test_predictions, use_test_cm = (
    evaluate_test_predictions(
        model_name="Universal Sentence Encoder",
        y_true=y_test,
        probabilities=use_test_probabilities,
        threshold=USE_THRESHOLD,
    )
)

print(
    "USE probability range: "
    f"{use_test_probabilities.min():.6f} "
    f"to {use_test_probabilities.max():.6f}"
)

print(f"\nUSE predictions: {len(use_test_probabilities)}")

print("\nUSE confusion matrix:")
print(use_test_cm)

pd.DataFrame([use_test_metrics]).round(4)

### 5.6 Final held-out model comparison

The table below compares all three frozen models on the held-out test set.

These results are not used to retune thresholds or alter model architectures. The Universal Sentence Encoder had already been identified as the leading candidate using validation data; the test results provide an independent estimate of its generalization performance.

For the spam-detection task, F1 and PR-AUC are particularly informative because the positive class is substantially less frequent than ham. False positives and false negatives are also reported because they correspond directly to different business costs.

In [ ]:
# ---------------------------------------------------------------------------
# Compare all frozen models on the held-out test set
# ---------------------------------------------------------------------------

final_test_results = pd.DataFrame(
    [
        baseline_test_metrics,
        gru_test_metrics,
        use_test_metrics,
    ]
)

final_test_results = final_test_results[
    [
        "model",
        "threshold",
        "accuracy",
        "precision",
        "recall",
        "f1",
        "roc_auc",
        "pr_auc",
        "false_positives",
        "false_negatives",
    ]
]

final_test_results.round(4)

### Interpretation

All three models achieve high overall accuracy, but accuracy alone is insufficient because ham messages substantially outnumber spam messages.

The **baseline model** achieves 97.03% accuracy and an F1-score of 0.8770. At its validation-selected threshold of 0.40, it produces 9 false positives and misses 14 spam messages. Its relatively high number of false positives is important because legitimate SMS messages incorrectly sent to spam may negatively affect users.

The **GRU model** achieves 97.80% accuracy and an F1-score of 0.9029. Its conservative validation-selected threshold of 0.89 results in perfect test precision: none of the 678 legitimate messages is classified as spam. However, this comes at the cost of lower recall (0.8229), with 17 of the 96 spam messages missed.

The **Universal Sentence Encoder model** provides the strongest overall balance. It achieves 98.19% accuracy, 0.9556 precision, 0.8958 recall, and the highest test F1-score (0.9247). It also obtains the highest PR-AUC (0.9678) and ROC-AUC (0.9854). Its confusion matrix contains only 4 false positives and 10 false negatives.

These results confirm the validation-based selection of the Universal Sentence Encoder as the final model. Compared with the baseline, it substantially reduces both false positives and false negatives. Compared with the GRU, it accepts four false positives in exchange for detecting seven additional spam messages.

The final choice therefore reflects both predictive performance and the business trade-off between blocking legitimate messages and allowing spam through.

In [ ]:
# ---------------------------------------------------------------------------
# Save final held-out test metrics
# ---------------------------------------------------------------------------

METRICS_DIR = PROJECT_ROOT / "outputs" / "metrics"
METRICS_DIR.mkdir(parents=True, exist_ok=True)

FINAL_METRICS_PATH = METRICS_DIR / "final_test_metrics.csv"

final_test_results.to_csv(
    FINAL_METRICS_PATH,
    index=False,
)

print(f"Final metrics saved to: {FINAL_METRICS_PATH}")
print(f"Metrics file exists:    {FINAL_METRICS_PATH.exists()}")

### 5.8 Confusion-matrix comparison

Confusion matrices provide a direct view of the business-relevant classification errors.

For this project:

- **false positive:** a legitimate ham message is incorrectly blocked as spam;
- **false negative:** a spam message is incorrectly allowed through.

The matrices below use the frozen thresholds selected during validation.

In [ ]:
# ---------------------------------------------------------------------------
# Visualize held-out confusion matrices for the three frozen models
# ---------------------------------------------------------------------------

import matplotlib.pyplot as plt
from sklearn.metrics import ConfusionMatrixDisplay

FIGURES_DIR = PROJECT_ROOT / "outputs" / "figures"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

confusion_matrices = {
    "Baseline — threshold 0.40": baseline_test_cm,
    "GRU — threshold 0.89": gru_test_cm,
    "USE — threshold 0.50": use_test_cm,
}

for model_name, cm in confusion_matrices.items():
    fig, ax = plt.subplots(figsize=(5, 4))

    # Use a specific variable name so that Jupyter's display()
    # function is not overwritten.
    cm_display = ConfusionMatrixDisplay(
        confusion_matrix=cm,
        display_labels=["Ham", "Spam"],
    )

    cm_display.plot(
        ax=ax,
        cmap="Blues",
        colorbar=False,
    )

    ax.set_title(model_name)
    plt.tight_layout()

    # Create a filesystem-safe filename from the model name.
    filename = (
        model_name.lower()
        .replace(" — ", "_")
        .replace(" ", "_")
        .replace(".", "_")
    )

    figure_path = FIGURES_DIR / f"{filename}_confusion_matrix.png"

    fig.savefig(
        figure_path,
        dpi=150,
        bbox_inches="tight",
    )

    plt.show()

    print(f"Saved: {figure_path}")

### 5.9 Final model error analysis

The Universal Sentence Encoder is the final selected model. Its remaining false positives and false negatives are inspected below to understand where the classifier still fails.

This analysis is descriptive only. No model changes or threshold adjustments are made based on test errors.

In [ ]:
# ---------------------------------------------------------------------------
# Inspect the final USE model's held-out classification errors
# ---------------------------------------------------------------------------

from IPython.display import display

use_error_analysis = test_df[["text", "label"]].copy()

use_error_analysis["true_label"] = y_test.to_numpy()
use_error_analysis["predicted_label"] = use_test_predictions
use_error_analysis["spam_probability"] = use_test_probabilities

use_false_positives = use_error_analysis[
    (use_error_analysis["true_label"] == 0)
    & (use_error_analysis["predicted_label"] == 1)
].copy()

use_false_negatives = use_error_analysis[
    (use_error_analysis["true_label"] == 1)
    & (use_error_analysis["predicted_label"] == 0)
].copy()

# Sort false positives from most to least confidently classified as spam.
use_false_positives = use_false_positives.sort_values(
    "spam_probability",
    ascending=False,
)

# Sort false negatives from most confidently ham-like upward.
use_false_negatives = use_false_negatives.sort_values(
    "spam_probability",
    ascending=True,
)

print(f"USE false positives: {len(use_false_positives)}")
print(f"USE false negatives: {len(use_false_negatives)}")

print("\nFalse positives:")
display(
    use_false_positives[
        ["text", "spam_probability"]
    ]
)

print("\nFalse negatives:")
display(
    use_false_negatives[
        ["text", "spam_probability"]
    ]
)

### Error-analysis interpretation

The final USE model makes 14 errors on the 774-message held-out test set: 4 false positives and 10 false negatives.

The **false positives** contain several characteristics that can plausibly resemble spam, including unusual formatting, capitalization, abbreviated language, and message structures that resemble announcements or promotional communication. Two false positives are close to the decision boundary (approximately 0.53 and 0.54), while two receive high spam probabilities (approximately 0.91 and 0.95). This indicates that some legitimate messages are semantically similar to patterns learned as spam rather than merely being threshold-edge cases.

The **false negatives** reveal a more important limitation. Several messages contain recognizable spam-like cues such as "FREE", ringtone offers, mobile-content language, chat services, or unsolicited promotional wording, yet the model assigns some of them very low spam probabilities. In particular, multiple false negatives receive probabilities below 0.01. Other errors lie much closer to the 0.50 threshold.

This mixture of high-confidence and borderline errors shows that threshold adjustment alone would not resolve the remaining mistakes. Lowering the threshold could recover some borderline spam messages, but would not recover several high-confidence false negatives and would simultaneously increase the number of legitimate messages classified as spam.

The remaining errors therefore appear to reflect limitations in the learned representation and classifier rather than a simple threshold-calibration problem. The validation-selected threshold of 0.50 is consequently retained unchanged.

### 5.10 Data-augmentation decision

Data augmentation was considered but deliberately excluded from the final modeling pipeline.

SMS spam classification is highly sensitive to short lexical and structural cues such as phone numbers, URLs, prices, promotional terms, abbreviations, punctuation, and calls to action. Generic NLP augmentation techniques such as random word deletion, synonym replacement, or word swapping could alter precisely the features that determine whether a message is spam and may therefore introduce label noise.

The experiments also do not provide strong evidence that augmentation is necessary. The final USE classifier generalizes from validation to the untouched test set with an F1-score of 0.9247 and a PR-AUC of 0.9678 despite the relatively small and imbalanced dataset.

For these reasons, no augmentation method was introduced without a domain-specific justification. A future extension could investigate controlled augmentation techniques, but they should be evaluated against the same unaugmented architecture to demonstrate that they improve generalization rather than simply increasing the amount of training data.

## Final conclusion

Three neural-network approaches were developed and evaluated progressively:

1. a word-embedding baseline with global average pooling;
2. a GRU sequence model;
3. a transfer-learning approach using frozen Universal Sentence Encoder representations.

Model architectures and operating thresholds were selected using training and validation data only. The held-out test set remained untouched until final evaluation.

At their frozen thresholds, the baseline, GRU, and USE models achieve test F1-scores of 0.8770, 0.9029, and 0.9247 respectively.

The **Universal Sentence Encoder classifier is selected as the final model**. On the held-out test set it achieves:

- accuracy: **0.9819**
- spam precision: **0.9556**
- spam recall: **0.8958**
- spam F1-score: **0.9247**
- ROC-AUC: **0.9854**
- PR-AUC: **0.9678**
- false positives: **4**
- false negatives: **10**

The GRU remains noteworthy because its conservative threshold produces zero false positives, but this comes at the cost of missing 17 spam messages. USE provides a stronger overall balance between protecting legitimate SMS messages and detecting unwanted spam.

The transfer-learning result also demonstrates the value of pretrained semantic representations for this relatively small text-classification dataset: the USE classifier achieves the strongest overall generalization while requiring only a small trainable classification head on top of the frozen 512-dimensional sentence representation.

The remaining USE errors show that some spam messages are classified with high confidence as ham, while some legitimate messages strongly resemble spam. These cases cannot be resolved through threshold adjustment alone and represent useful targets for future improvements in data quality, representation learning, or domain-specific modeling.